# Taxonomy-Match Classifier — Data Exploration & Decision-Support Charts

Companion notebook to the blocked taxonomy-matching investigation in
`docs/superpowers/plans/2026-07-18-taxonomy-match-classifier.md` and its predecessor
`docs/superpowers/plans/2026-07-17-embedding-nn-match.md`.

**What this is for:** every number quoted in that investigation (precision ~50-57%, feature importances,
the confidence-threshold sweep) was produced by agent-run scripts against BigQuery. This notebook reproduces
the same pipeline from scratch, inline, so you can look at the underlying data yourself — not just the
summary numbers — and decide whether the "blocked" conclusion holds up.

**Requirements:**
- A Google account with BigQuery access to project `sincere-hearth-273704` (the same access this repo's
  pipeline already uses).
- Run top to bottom. Each section is independent enough to re-run on its own once the data is loaded.

**Structure:**
1. Setup (auth, BigQuery client)
2. Load the training pairs (same query as `sql/queries/training_pairs.sql`)
3. Data sanity checks — read real examples, check for obvious data problems
4. Feature distributions by label (does any single feature actually separate correct from incorrect?)
5. Train the classifier (same as `script/train_taxonomy_matcher.py`)
6. Feature importance chart
7. Confidence-threshold sweep chart — the key chart for the auto-match decision
8. Precision across categories — is this one hard category, or a general ceiling?
9. Error inspection — read the actual misclassified pairs

## 1. Setup

In [ ]:
!pip install -q google-cloud-bigquery db-dtypes xgboost scikit-learn

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated. Next cell creates the BigQuery client.')

In [ ]:
PROJECT = "sincere-hearth-273704"

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT)
print(f"BigQuery client ready for project {PROJECT}")

## 2. Load the training pairs

This is the exact query committed at `sql/queries/training_pairs.sql` (as of the `product_line_match`
follow-up experiment) — one row per (product, candidate taxonomy entry) pair: the true positive plus its
top-5 nearest same-brand non-matches by embedding distance (the confusable pairs that break naive matching).
Covers all 20 TH categories plus `shopee_sg_toothpaste`.

In [ ]:
TRAINING_PAIRS_SQL = """
WITH latest_month AS (
  SELECT MAX(month) AS month FROM `sincere-hearth-273704.magpie.marketshare_universe_niq`
),
anchors AS (
  SELECT
    m.product_id, m.platform, m.country, m.master_table, m.taxonomy_id AS assigned_taxonomy_id,
    u.sku_name, u.brand_id, ue.embedding AS product_embedding
  FROM `sincere-hearth-273704.magpie_reference.product_taxonomy_map` m
  JOIN `sincere-hearth-273704.magpie.marketshare_universe_niq` u
    ON u.product_id = m.product_id AND u.ecommerce_platform = m.platform AND u.country = m.country
  CROSS JOIN latest_month
  JOIN `sincere-hearth-273704.magpie_reference.universe_sku_embeddings` ue
    ON ue.product_id = m.product_id AND ue.platform = m.platform AND ue.country = m.country
  WHERE u.month = latest_month.month
    AND m.source = 'LLM'
    AND (m.master_table LIKE 'shopee_th_%' OR m.master_table = 'shopee_sg_toothpaste')
    AND u.brand_confidence IN ('HIGH', 'MEDIUM')
    AND u.brand_id NOT IN ('BRD-UNDEFINED', 'BRD-UNBRANDED')
),
anchors_parsed AS (
  SELECT a.*, parsed.size_text AS parsed_size, bd.canonical_name AS brand_canonical
  FROM anchors a
  CROSS JOIN UNNEST([`sincere-hearth-273704.magpie_reference.parse_size`(a.sku_name)]) AS parsed
  JOIN `sincere-hearth-273704.magpie_reference.brand_dict` bd ON bd.brand_id = a.brand_id
),
candidates AS (
  SELECT
    a.product_id, a.platform, a.country, a.master_table, a.sku_name, a.assigned_taxonomy_id,
    a.parsed_size, a.brand_canonical, a.brand_id,
    pt.taxonomy_id AS candidate_taxonomy_id,
    pt.size AS candidate_size, pt.pack_count AS candidate_pack_count,
    pt.product_line AS candidate_product_line,
    pt.canonical_name AS candidate_canonical_name,
    ML.DISTANCE(a.product_embedding, pte.embedding, 'COSINE') AS embedding_cosine_distance
  FROM anchors_parsed a
  JOIN `sincere-hearth-273704.magpie_reference.product_taxonomy` pt ON pt.brand_id = a.brand_id
  JOIN `sincere-hearth-273704.magpie_reference.product_taxonomy_embeddings` pte
    ON pte.taxonomy_id = pt.taxonomy_id
),
labeled AS (
  SELECT
    *,
    IF(candidate_taxonomy_id = assigned_taxonomy_id, 1, 0) AS label,
    ROW_NUMBER() OVER (
      PARTITION BY product_id, platform, country
      ORDER BY IF(candidate_taxonomy_id = assigned_taxonomy_id, 0, 1), embedding_cosine_distance ASC
    ) AS rn
  FROM candidates
)
SELECT
  l.product_id, l.platform, l.country, l.master_table, l.candidate_taxonomy_id AS taxonomy_id, l.label,
  l.sku_name, l.candidate_canonical_name,
  l.embedding_cosine_distance,
  IF(l.parsed_size IS NULL OR l.candidate_size IS NULL, 'unknown',
     IF(l.parsed_size = l.candidate_size, 'match', 'mismatch')) AS size_match,
  REGEXP_EXTRACT(LOWER(l.sku_name), 'x\\\\s*(\\\\d+)') AS pack_multiplier_signal,
  l.candidate_pack_count,
  EDIT_DISTANCE(
    LOWER(REPLACE(REPLACE(l.sku_name, l.brand_canonical, ''), IFNULL(l.parsed_size, ''), '')),
    LOWER(REPLACE(REPLACE(l.candidate_canonical_name, l.brand_canonical, ''), IFNULL(l.parsed_size, ''), ''))
  ) AS edit_distance_stripped,
  LOWER(REPLACE(REPLACE(l.sku_name, l.brand_canonical, ''), IFNULL(l.parsed_size, ''), '')) AS sku_text_stripped,
  LOWER(REPLACE(REPLACE(l.candidate_canonical_name, l.brand_canonical, ''), IFNULL(l.parsed_size, ''), '')) AS candidate_text_stripped,
  IFNULL(bvk.variant, '') != '' AS keyword_table_hit,
  SAFE_DIVIDE(LENGTH(l.sku_name), LENGTH(l.candidate_canonical_name)) AS text_length_ratio,
  IF(l.candidate_product_line IS NOT NULL AND STRPOS(LOWER(l.sku_name), LOWER(l.candidate_product_line)) > 0, TRUE, FALSE) AS product_line_match
FROM labeled l
LEFT JOIN `sincere-hearth-273704.magpie_reference.brand_variant_keywords` bvk
  ON bvk.brand_id = l.brand_id
  AND STRPOS(LOWER(l.sku_name), LOWER(bvk.variant)) > 0
WHERE l.rn <= 6
"""

df = client.query(TRAINING_PAIRS_SQL).to_dataframe()
print(f"Loaded {len(df):,} (product, candidate) pairs, {df['label'].sum():,} positive")
df.head(10)

## 3. Data sanity checks — does the data look right?

Before trusting any precision number, look at the raw material it's built from.

In [ ]:
# Row counts and positive rate per category — is any one category dominating or oddly shaped?
summary = df.groupby('master_table').agg(
    products=('product_id', 'nunique'),
    pairs=('product_id', 'count'),
    positive_rate=('label', 'mean'),
).sort_values('products', ascending=False)
summary

In [ ]:
# Candidate pool size per product - how many same-brand candidates does each product actually face?
import matplotlib.pyplot as plt

pool_sizes = df.groupby(['product_id', 'platform', 'country']).size()
fig, ax = plt.subplots(figsize=(8, 4))
pool_sizes.value_counts().sort_index().plot(kind='bar', ax=ax, color='#4C72B0')
ax.set_xlabel('Candidates per product (capped at 6 by the query)')
ax.set_ylabel('Number of products')
ax.set_title('Candidate pool size distribution')
plt.tight_layout()
plt.show()
print('If most products only ever have 1 candidate, matching is trivial by construction; if most have 6,')
print('every product genuinely has to be discriminated against 5 real alternatives.')

In [ ]:
# Read real examples — the actual thing to eyeball for "is this data sane"
pd = __import__('pandas')
pd.set_option('display.max_colwidth', 80)
sample_cols = ['master_table', 'sku_name', 'candidate_canonical_name', 'label', 'size_match', 'candidate_pack_count', 'embedding_cosine_distance']
df[sample_cols].sample(20, random_state=42)

## 4. Feature engineering

Mirrors `script/taxonomy_match_encoding.py` exactly — the single source of truth in the repo for turning
raw query output into model-ready features. Reproduced here so this notebook has no dependency on the repo.

In [ ]:
import re

FEATURE_COLUMNS = [
    "embedding_cosine_distance",
    "edit_distance_stripped",
    "token_jaccard_stripped",
    "keyword_table_hit",
    "text_length_ratio",
    "size_match_code",
    "pack_signal_present",
    "pack_match_code",
    "product_line_match_code",
]

def token_jaccard(a, b):
    tokens_a = set(re.findall(r"\w+", str(a)))
    tokens_b = set(re.findall(r"\w+", str(b)))
    if not tokens_a or not tokens_b:
        return 0.0
    return len(tokens_a & tokens_b) / len(tokens_a | tokens_b)

def encode_features(df):
    df = df.copy()
    df["token_jaccard_stripped"] = df.apply(
        lambda r: token_jaccard(r["sku_text_stripped"], r["candidate_text_stripped"]), axis=1
    )
    df["size_match_code"] = df["size_match"].map({"match": 1, "mismatch": -1, "unknown": 0})
    df["keyword_table_hit"] = df["keyword_table_hit"].astype(int)
    df["pack_signal_present"] = df["pack_multiplier_signal"].notna().astype(int)
    df["pack_match_code"] = 0
    has_both = df["pack_multiplier_signal"].notna() & df["candidate_pack_count"].notna()
    df.loc[has_both, "pack_match_code"] = (
        df.loc[has_both, "pack_multiplier_signal"].astype(float)
        == df.loc[has_both, "candidate_pack_count"].astype(float)
    ).astype(int) * 2 - 1
    df["product_line_match_code"] = df["product_line_match"].astype(int)
    return df

df = encode_features(df)
print('Features encoded. Missing-value rate per feature (should mostly be 0 - NaNs mean a join failed):')
df[FEATURE_COLUMNS].isna().mean().sort_values(ascending=False)

## 5. Feature distributions by label

For each feature: does it actually look different between correct (label=1) and incorrect (label=0)
candidates? A feature that overlaps almost completely between the two groups isn't doing much work,
regardless of what a trained model's importance score later says.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, col in zip(axes.flat, ['embedding_cosine_distance', 'edit_distance_stripped', 'token_jaccard_stripped', 'text_length_ratio']):
    for label_val, color, name in [(1, '#55A868', 'correct match'), (0, '#C44E52', 'wrong candidate')]:
        subset = df.loc[df['label'] == label_val, col].dropna()
        ax.hist(subset, bins=40, alpha=0.5, density=True, color=color, label=name)
    ax.set_title(col)
    ax.legend()

plt.tight_layout()
plt.suptitle('Feature distributions: correct match vs. wrong candidate', y=1.02, fontsize=14)
plt.show()

In [ ]:
# Categorical/boolean features: match rate by label
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ['size_match', 'keyword_table_hit', 'product_line_match']):
    ct = df.groupby('label')[col].value_counts(normalize=True).unstack()
    ct.T.plot(kind='bar', ax=ax, color=['#C44E52', '#55A868'])
    ax.set_title(col)
    ax.legend(['wrong candidate', 'correct match'])
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 6. Train the classifier

Same approach as `script/train_taxonomy_matcher.py`: train on every category except one held-out eval
category, evaluate top-1-by-probability precision on the held-out set. **Change `EVAL_CATEGORY` below to
try any of the categories in the data** (see the Section 3 table for what's available).

In [ ]:
EVAL_CATEGORY = "shopee_sg_toothpaste"  # try e.g. shopee_th_toothpaste, shopee_th_shampoo, shopee_th_coffee

import xgboost as xgb

train_df = df[df["master_table"] != EVAL_CATEGORY]
eval_df = df[df["master_table"] == EVAL_CATEGORY].copy()

model = xgb.XGBClassifier(n_estimators=200, max_depth=4, eval_metric="logloss")
model.fit(train_df[FEATURE_COLUMNS], train_df["label"])

eval_df["predicted_prob"] = model.predict_proba(eval_df[FEATURE_COLUMNS])[:, 1]
top1 = eval_df.loc[eval_df.groupby(["product_id", "platform", "country"])["predicted_prob"].idxmax()]

raw_precision = (top1["label"] == 1).mean()
print(f"Held-out {EVAL_CATEGORY}: {len(top1)} products")
print(f"Raw top-1 precision: {raw_precision:.4f}")

In [ ]:
# Ground-truth hygiene check: does the recorded ground-truth size even agree with the product's own parsed size?
# (Same self-consistency check used throughout the investigation - some "errors" turn out to be bad ground truth,
# not bad matching. Restricting to the clean subset gives a second, more trustworthy precision number.)
CLEAN_SQL = f"""
    SELECT DISTINCT m.product_id
    FROM `{PROJECT}.magpie_reference.product_taxonomy_map` m
    JOIN `{PROJECT}.magpie.marketshare_universe_niq` u
      ON u.product_id = m.product_id AND u.ecommerce_platform = m.platform AND u.country = m.country
    JOIN `{PROJECT}.magpie_reference.product_taxonomy` pt ON pt.taxonomy_id = m.taxonomy_id
    CROSS JOIN UNNEST([`{PROJECT}.magpie_reference.parse_size`(u.sku_name)]) AS parsed
    WHERE u.month = (SELECT MAX(month) FROM `{PROJECT}.magpie.marketshare_universe_niq`)
      AND m.master_table = '{EVAL_CATEGORY}' AND m.source = 'LLM'
      AND (parsed.size_text IS NULL OR pt.size IS NULL OR parsed.size_text = pt.size)
"""
clean_ids = set(r.product_id for r in client.query(CLEAN_SQL).result())
clean_subset = top1[top1["product_id"].isin(clean_ids)]
clean_precision = (clean_subset["label"] == 1).mean()
print(f"Ground-truth-clean subset precision: {clean_precision:.4f} (n={len(clean_subset)})")

## 7. Feature importance

In [ ]:
importances = dict(zip(FEATURE_COLUMNS, model.feature_importances_))
imp_series = __import__('pandas').Series(importances).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
imp_series.plot(kind='barh', ax=ax, color='#4C72B0')
ax.set_xlabel('Feature importance (gain-normalized)')
ax.set_title(f'What the model actually uses (held out: {EVAL_CATEGORY})')
plt.tight_layout()
plt.show()

## 8. Confidence-threshold sweep — the key chart for the auto-match decision

The whole point of a probability output is to only auto-match the *confident* predictions. This chart is
the real test: as the threshold rises, does precision climb to something usable (e.g. 0.98) while keeping
a meaningful fraction of products covered? Or does confidence and correctness not track each other?

In [ ]:
thresholds = [0.0, 0.3, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 0.97, 0.98, 0.99]
rows = []
for t in thresholds:
    subset = top1[top1["predicted_prob"] >= t]
    n = len(subset)
    coverage = n / len(top1)
    precision = (subset["label"] == 1).mean() if n else float("nan")
    rows.append({"threshold": t, "n": n, "coverage": coverage, "precision": precision})

sweep_df = __import__('pandas').DataFrame(rows)
display(sweep_df)

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(sweep_df["threshold"], sweep_df["precision"], marker="o", color="#C44E52", label="Precision")
ax1.axhline(0.98, color="#C44E52", linestyle="--", alpha=0.5, label="0.98 target")
ax1.set_xlabel("Confidence threshold (min predicted probability)")
ax1.set_ylabel("Precision", color="#C44E52")
ax1.set_ylim(0, 1.05)

ax2 = ax1.twinx()
ax2.plot(sweep_df["threshold"], sweep_df["coverage"], marker="s", color="#4C72B0", label="Coverage")
ax2.set_ylabel("Coverage (fraction of products with a prediction at/above threshold)", color="#4C72B0")
ax2.set_ylim(0, 1.05)

fig.suptitle(f"Precision vs. coverage by confidence threshold ({EVAL_CATEGORY})")
fig.legend(loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=2)
plt.tight_layout()
plt.show()
print('Look for: is there ANY threshold with both high precision (near the dashed line) AND non-trivial coverage?')

## 9. Precision across categories — is this one hard category, or a general ceiling?

Re-trains once per held-out category (a few seconds each). If precision is consistently in the same band
regardless of category or country, that's evidence of a general modeling ceiling, not a quirk of one dataset.

In [ ]:
candidate_categories = [c for c in df['master_table'].unique() if df[df['master_table']==c]['product_id'].nunique() >= 200]
print('Categories with enough held-out volume to be meaningful:', candidate_categories)

category_results = []
for cat in candidate_categories:
    tr = df[df["master_table"] != cat]
    ev = df[df["master_table"] == cat].copy()
    m = xgb.XGBClassifier(n_estimators=200, max_depth=4, eval_metric="logloss")
    m.fit(tr[FEATURE_COLUMNS], tr["label"])
    ev["predicted_prob"] = m.predict_proba(ev[FEATURE_COLUMNS])[:, 1]
    t1 = ev.loc[ev.groupby(["product_id", "platform", "country"])["predicted_prob"].idxmax()]
    category_results.append({"category": cat, "n_products": len(t1), "precision": (t1["label"] == 1).mean()})

cat_df = __import__('pandas').DataFrame(category_results).sort_values("precision")
display(cat_df)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(cat_df["category"], cat_df["precision"], color="#4C72B0")
ax.axvline(0.98, color="#C44E52", linestyle="--", label="0.98 target")
ax.set_xlabel("Held-out top-1 precision")
ax.set_title("Precision by held-out category")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Error inspection — read the actual misclassified pairs

The single most useful sanity check: read real disagreements yourself. Is the model wrong, or is the
recorded ground truth wrong? (Both happened in the underlying investigation — see the plan's Appendix.)

In [ ]:
errors = top1[top1["label"] == 0].merge(
    df[df["master_table"] == EVAL_CATEGORY][["product_id", "platform", "country", "taxonomy_id", "label", "candidate_canonical_name"]].query("label == 1"),
    on=["product_id", "platform", "country"], suffixes=("_predicted", "_actual")
)
pd.set_option('display.max_colwidth', 60)
cols = ['sku_name', 'candidate_canonical_name_predicted', 'candidate_canonical_name_actual', 'predicted_prob']
errors[cols].sample(min(20, len(errors)), random_state=1)

## Notes

- Full narrative context, prior results (v1 embedding-only ranking, this v2 classifier, the `product_line` +
  `sg_toothpaste` follow-up), and the decision history are in `docs/superpowers/plans/2026-07-18-taxonomy-match-classifier.md`
  and `docs/traditional-ml-execution-model.md` in the repo.
- If Section 8's chart shows no threshold with both high precision and usable coverage, and Section 9 shows
  the same band across categories/countries, that's the empirical basis for treating this feature set as
  tapped out — the next lever is a qualitatively different signal (product images), not another engineered
  text feature.